# Solar filament segmentation - inference notebook

Thin wrapper around `scripts/run_pipeline.py`, which reproduces the public-0.40 submission:

1. **Detect** - YOLO11m-seg at 1536 px, three passes (identity, horizontal flip, vertical flip), each caching per-instance soft mask probabilities.
2. **Fuse** - `filseg.ensemble.ensemble_caches` merges the three passes' instances (IoU 0.5) into one cache.
3. **Refine** - two crop refiners re-predict each instance mask from the image crop + the fused soft mask as prior: a U-Net and a SAM ViT-B + LoRA model.
4. **Blend** - the two refined soft masks are averaged 50/50 (`scripts/blend_caches.py`).
5. **Post-process** - calibrated parameters (`calib.json`: conf 0.25, min_area 100, max_det 20, nms 0.3, mask_thr 0.35), overlap resolution, RLE writing and validation.

Run from a checkout that contains the weights (they are not in git, see README).

## 1. Setup
Install pinned requirements (skip on a prepared environment) and set the paths.

In [ ]:
import subprocess, sys
from pathlib import Path

# On Kaggle / a fresh machine. Comment out if the environment is already set up.
# !{sys.executable} -m pip install -q -r requirements.txt

REPO = Path('..').resolve() if Path('../scripts/run_pipeline.py').exists() else Path('.').resolve()
IMAGES = REPO / 'data/raw/MAGFiLO_1.0_Kaggle_2026/test/test_images'   # directory of *.jpeg
OUT = Path('submission.csv').resolve()
WORKDIR = Path('pipeline_work').resolve()                           # intermediate caches
DEVICE = 'cuda'
assert (REPO / 'scripts/run_pipeline.py').exists(), REPO
print(REPO, len(list(IMAGES.glob('*.jpeg'))), 'images')

## 2. Run the pipeline
One command runs stages 1-5. Each intermediate cache (`raw.pkl`, `raw_h.pkl`, `raw_v.pkl`, `flip3.pkl`, `unet.pkl`, `sam.pkl`, `blend.pkl`) is written to `WORKDIR`; an existing cache is reused unless `--force` is given. Weight paths default to the repo layout (`--yolo`, `--unet`, `--sam`, `--calib` override them). The SAM refiner also needs the `facebook/sam-vit-base` base model from the Hugging Face cache/hub.

In [ ]:
cmd = [sys.executable, 'scripts/run_pipeline.py', '--images', str(IMAGES), '--out', str(OUT),
       '--workdir', str(WORKDIR), '--device', DEVICE]
subprocess.run(cmd, cwd=REPO, check=True)

## 3. Inspect the result
The pipeline already validates the CSV (all test stems present, valid RLEs, no overlapping instances).

In [ ]:
import pandas as pd
sub = pd.read_csv(OUT)
print(sub.shape)
sub['stem'] = sub['filament_id'].str.rsplit('_', n=1).str[0]
print(sub['stem'].nunique(), 'images with predictions; mean', round(len(sub) / sub['stem'].nunique(), 2), 'instances per image')
sub.head()